In [11]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ['TORCH_USE_CUDA_DSA'] = "1"
import pandas as pd
train_df = pd.read_parquet(r"final_data\chunk-based-split-3\no_scaler_train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"final_data\chunk-based-split-3\no_scaler_valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"final_data\chunk-based-split-3\no_scaler_test_df_prepared.parquet", engine="pyarrow")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)



In [12]:
import pandas as pd
import numpy as np
import torch
import gc
from torch_geometric.data import Data # thư viên Data giúp lưu trữ đồ thị lưới 
from tqdm.notebook import tqdm
import os

# bật cảnh báo trùng lặp của KMP (Intel MKL) để tránh lỗi crash khi dùng nhiều luồng trên CPU
os.environ["KMP_DUPLICATE_BLOCK_WARNING"] = "TRUE"

# Xây dựng đồ thị Flow-based Graph có đặc trưng cạnh
def build_ip_topology_graph(df, train_mask, valid_mask, test_mask, window_size=10):
    print(f"Bước 1: Trích xuất Node Features (X) và Label (y)...")
    
    cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port', 'label']
    
    # các cột cần drop 
    cols_present = [c for c in cols_to_drop if c in df.columns]

    # các cột còn lại sẽ là feature
    feature_cols = [c for c in df.columns if c not in cols_present]
    
    # tạo một numpy array chứa các feature
    features_np = np.empty((len(df), len(feature_cols)), dtype=np.float32)
    for i, col in enumerate(feature_cols):
        features_np[:, i] = df[col].astype(np.float32).values
        
    # --- ĐÃ COMMENT VÔ HIỆU HOÁ SCALER TRONG HÀM ĐỂ XỬ LÝ ĐỒNG BỘ NGOÀI DATAFRAME ---
    # print("Bước 1.5: Áp dụng QuantileTransformer (Chỉ fit trên Train Mask để chống Data Leakage)...")
    # from sklearn.preprocessing import QuantileTransformer
    # scaler = QuantileTransformer(output_distribution='normal', random_state=42)
    
    # train_idx = np.where(train_mask.numpy())[0]
    # valid_idx = np.where(valid_mask.numpy())[0]
    # test_idx = np.where(test_mask.numpy())[0]
    
    # if len(train_idx) > 0:
    #     features_np[train_idx] = scaler.fit_transform(features_np[train_idx])
    # if len(valid_idx) > 0:
    #     features_np[valid_idx] = scaler.transform(features_np[valid_idx])
    # if len(test_idx) > 0:
    #     features_np[test_idx] = scaler.transform(features_np[test_idx])
    
    # ascontiguousarray: đảm bảo dữ liệu được lưu trữ liên tục trong bộ nhớ
    features_np = np.ascontiguousarray(features_np)
    
    # lấy node features (x_tensor) và label (y_tensor)
    x_tensor = torch.from_numpy(features_np).contiguous()
    y_tensor = torch.tensor(df['label'].values, dtype=torch.long).contiguous()

    print(f"Bước 2: Xây Dựng Đồ Thị Dây Chuyền Theo IP (Temporal IP Topology)...")

    # lấy các giá trị để xây dựng cạnh
    timestamps = df['timestamp_num'].values
    src_ips = df['src_ip'].values
    dst_ips = df['dst_ip'].values
    src_ports = df['src_port'].astype(str).values
    dst_ports = df['dst_port'].astype(str).values
    
    # tạo một mảng split_id để đánh dấu tập Train (0), Valid (1), Test (2)
    split_id = np.zeros(len(df), dtype=np.int8)
    split_id[valid_mask.numpy()] = 1
    split_id[test_mask.numpy()] = 2

    # gom nhóm các gói tin theo ip để nối cạnh
    ip_to_indices = {}
    for idx in tqdm(range(len(df)), desc="[1/2] Gom nhóm Gói tin theo IP"):
        s_ip = src_ips[idx]
        d_ip = dst_ips[idx]
        
        if s_ip not in ip_to_indices: ip_to_indices[s_ip] = []
        if len(ip_to_indices[s_ip]) == 0 or ip_to_indices[s_ip][-1] != idx:
            ip_to_indices[s_ip].append(idx)
            
        if d_ip not in ip_to_indices: ip_to_indices[d_ip] = []
        if len(ip_to_indices[d_ip]) == 0 or ip_to_indices[d_ip][-1] != idx:
            ip_to_indices[d_ip].append(idx)
            
    all_src, all_dst, all_edge_attrs = [], [], []
    
    # nối cạnh giữa các gói tin có cùng IP (src hoặc dst) trong một cửa sổ thời gian nhất định (window_size)
    for ip, indices in tqdm(ip_to_indices.items(), desc=f"[2/2] Căng lưới đồ thị (Window = {window_size})"):
        n_flows = len(indices)
        if n_flows < 2: continue
        
        for i in range(1, n_flows):
            curr_idx = indices[i]
            start_w = max(0, i - window_size) # start_w: chỉ số bắt đầu của cửa sổ (window)
            
            for j in range(start_w, i):
                past_idx = indices[j]
                
                dt_raw = abs(timestamps[curr_idx] - timestamps[past_idx])
                if dt_raw > 60.0:
                    continue
                
                dt = np.log1p(dt_raw * 1e6) / 18.0
                same_dir = 1.0 if src_ips[curr_idx] == src_ips[past_idx] else 0.0
                same_sport = 1.0 if src_ports[curr_idx] == src_ports[past_idx] else 0.0
                same_dport = 1.0 if dst_ports[curr_idx] == dst_ports[past_idx] else 0.0

                attr = [dt, same_dir, same_sport, same_dport]
                
                # Trả lại phương pháp nối cạnh siêu bảo mật chống Leakage giống hệt Notebook cũ
                # nếu 2 gói đều thuộc tập train
                if split_id[curr_idx] == 0 and split_id[past_idx] == 0:
                    # nối 2 chiều giữa curr_idx và past_idx
                    all_src.extend([curr_idx, past_idx])
                    all_dst.extend([past_idx, curr_idx])
                    all_edge_attrs.extend([attr, attr]) 
                else:
                    # nối 1 chiều theo thời gian đối với các gói tin thuộc tập valid/test (Quá khứ -> Tương lai)
                    all_src.append(past_idx)
                    all_dst.append(curr_idx)
                    all_edge_attrs.append(attr) 
                    
    print("\nHợp nhất và tạo Cạnh (Culling Edge Index)...")
    
    # tạo dataframe tạm thời để bỏ cạnh trùng lặp
    df_edges = pd.DataFrame({
        'src': all_src, 
        'dst': all_dst,
        'attr0': [a[0] for a in all_edge_attrs],
        'attr1': [a[1] for a in all_edge_attrs],
        'attr2': [a[2] for a in all_edge_attrs],
        'attr3': [a[3] for a in all_edge_attrs]
    })
    
    # loại bỏ các cạnh trùng lặp
    df_edges = df_edges.drop_duplicates(subset=['src', 'dst']).reset_index(drop=True)
    
    edge_index = torch.tensor(df_edges[['src', 'dst']].values.T, dtype=torch.long).contiguous()
    edge_attr = torch.tensor(df_edges[['attr0', 'attr1', 'attr2', 'attr3']].values, dtype=torch.float32).contiguous()
    
    print(f"HOÀN THIỆN ĐỒ THỊ TEMPORAL IP (CÓ EDGE FEATURES)!")
    print(f"   - Số Nodes : {x_tensor.size(0):,}")
    print(f"   - Số Edges : {edge_index.size(1):,}")
    print(f"   - Số Features/Cạnh : {edge_attr.shape[1]}")
    
    del features_np, df_edges, all_src, all_dst, all_edge_attrs
    gc.collect()
    
    # tạo đồ thị PyG Data
    graph = Data(x=x_tensor, edge_index=edge_index, edge_attr=edge_attr, y=y_tensor)
    
    # gắn mask
    graph.train_mask = train_mask
    graph.valid_mask = valid_mask
    graph.test_mask = test_mask
    return graph

In [13]:
import torch
import torch.nn.functional as F
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import GATv2Conv
from torch_geometric.utils import dropout_edge
import torch.nn as nn
from tqdm.notebook import tqdm
import copy
import gc
import numpy as np

# định nghĩa gat với 2 lớp
class GAT_Embedder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, embedding_dim, num_classes, heads=4, edge_dropout=0.1, edge_dim=4):
        super(GAT_Embedder, self).__init__()
        self.edge_dropout = edge_dropout 
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=heads, dropout=0.1, edge_dim=edge_dim)
        self.bn1 = nn.BatchNorm1d(hidden_channels * heads)
        self.conv2 = GATv2Conv(hidden_channels * heads, embedding_dim, heads=1, concat=False, dropout=0.1, edge_dim=edge_dim)
        self.bn2 = nn.BatchNorm1d(embedding_dim)
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x, edge_index, edge_attr):
        edge_index_dp, edge_mask = dropout_edge(edge_index, p=self.edge_dropout, force_undirected=False, training=self.training)
        edge_attr_dp = edge_attr[edge_mask] if edge_attr is not None else None
        
        x = F.dropout(x, p=0.25, training=self.training)
        x = self.conv1(x, edge_index_dp, edge_attr=edge_attr_dp)
        x = self.bn1(x)
        x = F.elu(x)
        
        edge_index_dp2, edge_mask2 = dropout_edge(edge_index, p=self.edge_dropout, force_undirected=False, training=self.training)
        edge_attr_dp2 = edge_attr[edge_mask2] if edge_attr is not None else None
        
        x = F.dropout(x, p=0.25, training=self.training)
        embedding = self.conv2(x, edge_index_dp2, edge_attr=edge_attr_dp2)
        embedding = self.bn2(embedding)
        embedding = F.elu(embedding) 
        
        out = self.classifier(embedding)
        return out, embedding

def train_gat(model, train_loader, valid_loader, device, epochs=5, lr=0.005, class_weights=None, patience=4):
    from sklearn.metrics import f1_score

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-3) 
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=2, verbose=True
    )
    
    if class_weights is not None:
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    else:
        criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
        
    best_val_f1 = -1.0
    best_model_weights = None
    no_improve_epochs = 0
    
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        total_train_nodes = 0
        all_train_preds = []
        all_train_labels = []
        
        pbar_train = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for batch in pbar_train:
            batch = batch.to(device)
            optimizer.zero_grad()
            
            out, emb = model(batch.x, batch.edge_index, batch.edge_attr)
            batch_size = batch.batch_size
            
            loss = criterion(out[:batch_size], batch.y[:batch_size])
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            
            optimizer.step()
            
            pred = out[:batch_size].argmax(dim=1)
            y_true = batch.y[:batch_size]
            
            all_train_preds.append(pred.cpu().detach().numpy())
            all_train_labels.append(y_true.cpu().detach().numpy())
            
            total_train_loss += loss.item() * batch_size
            total_train_nodes += batch_size
            
            pbar_train.set_postfix({'Loss': f"{loss.item():.4f}"})
            
            del batch, out, emb, loss, pred, y_true
            
        avg_train_loss = total_train_loss / total_train_nodes
        train_f1 = f1_score(np.concatenate(all_train_labels), np.concatenate(all_train_preds), average='macro')
        
        pbar_train.close()
            
        model.eval()
        total_val_loss = 0
        total_val_nodes = 0
        all_val_preds = []
        all_val_labels = []
        
        with torch.no_grad():
            pbar_val = tqdm(valid_loader, desc=f"Epoch {epoch+1}/{epochs} [Valid]")
            for batch in pbar_val:
                batch = batch.to(device)
                
                out, emb = model(batch.x, batch.edge_index, batch.edge_attr)
                batch_size = batch.batch_size
                
                loss = criterion(out[:batch_size], batch.y[:batch_size])
                pred = out[:batch_size].argmax(dim=1)
                y_true = batch.y[:batch_size]
                
                all_val_preds.append(pred.cpu().detach().numpy())
                all_val_labels.append(y_true.cpu().detach().numpy())
                
                total_val_loss += loss.item() * batch_size
                total_val_nodes += batch_size
                
                pbar_val.set_postfix({'Loss': f"{loss.item():.4f}"})
                
                del batch, out, emb, loss, pred, y_true
                
        pbar_val.close()
                
        avg_val_loss = total_val_loss / total_val_nodes
        val_f1 = f1_score(np.concatenate(all_val_labels), np.concatenate(all_val_preds), average='macro')
        
        scheduler.step(val_f1)
        
        print(f"=== Epoch {epoch+1} Tổng kết ===")
        print(f"   Train Loss: {avg_train_loss:.4f} | Train Macro F1: {train_f1:.4f}")
        print(f"   Valid Loss: {avg_val_loss:.4f}   | Valid Macro F1: {val_f1:.4f}")
        current_lr = optimizer.param_groups[0]['lr']
        print(f"   Learning Rate hiện tại: {current_lr:.6f}")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            no_improve_epochs = 0
            # CHUYỂN TRỌNG SỐ SANG CPU ĐỂ CHỐNG TÍCH TỤ VRAM:
            best_model_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f"Best model. Đã lưu trọng số tốt nhất tại Epoch {epoch+1} (Val Macro F1: {best_val_f1:.4f} | Val Loss: {avg_val_loss:.4f})")
        else:
            no_improve_epochs += 1
            print(f"Báo động Early Stopping: Không cải thiện F1 {no_improve_epochs}/{patience} epochs.")
            if no_improve_epochs >= patience:
                print(f"\nĐã kích hoạt Early Stopping tại Epoch {epoch+1}! Lùi về checkpoint tốt nhất.")
                break
                
        # ĐÃ XÓA LỆNH torch.cuda.empty_cache() TẠI ĐÂY ĐỂ CHỐNG VRAM SWAPPING PENALTY!

    if best_model_weights is not None:
        # Load lại từ CPU lên GPU
        best_model_weights_gpu = {k: v.to(device) for k, v in best_model_weights.items()}
        model.load_state_dict(best_model_weights_gpu)
        print(f"\nHoàn tất Train! Đã load lại weights MANG LẠI F1 CAO NHẤT (Val Macro F1: {best_val_f1:.4f}) để dùng trích xuất Embedding!")
        
    # Xả một lần duy nhất lúc hoàn tất bộ hàm
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    return model

# hàm trích xuất embedding từ model GAT
@torch.no_grad()
def extract_embeddings_mask(model, graph_data, mask, device='cpu'):
    model.eval()
    print("Đang khởi tạo Inference Loader lấy Embeddings...")
    
    # loader dành cho lúc test/extract
    loader = NeighborLoader(
        graph_data,
        num_neighbors=[10, 5], 
        input_nodes=mask,
        batch_size=512,
        shuffle=False,        
        num_workers=0 
    )
    
    all_embeddings = []
    all_labels = []
    
    pbar = tqdm(loader, desc="Đang trích xuất Vectơ Embedding")
    for batch in pbar:
        batch = batch.to(device)
        
        # truyền qua model để lấy embedding
        _, emb = model(batch.x, batch.edge_index, batch.edge_attr)
        
        # lấy embedding của các node trong batch
        bs = batch.batch_size
        all_embeddings.append(emb[:bs].cpu().numpy())
        all_labels.append(batch.y[:bs].cpu().numpy())
        
        # xóa đi batch, emb để tránh nghẽn RAM
        del batch, emb, _
        
    pbar.close()
    
    # dọn rác 
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    # hợp lại thành ma trận
    final_embeddings = np.concatenate(all_embeddings, axis=0)
    final_labels = np.concatenate(all_labels, axis=0)
    
    print(f"Trích xuất thành công! Kích thước Ma trận Embeddings: {final_embeddings.shape}")
    return final_embeddings, final_labels

# hàm huấn luyện và đánh giá XGBoost từ embedding của GAT
def train_evaluate_xgboost(X_train_emb, y_train, X_valid_emb, y_valid, X_test_emb, y_test):
    import xgboost as xgb
    from sklearn.metrics import classification_report, accuracy_score, f1_score
    from sklearn.utils.class_weight import compute_class_weight
    
    print("\n--- BẮT ĐẦU TRAIN XGBOOST TỪ BASE GAT EMBEDDING ---")

    # tính toán class weights cho XGBoost dựa trên phân bố lớp tập train
    print("Đang tính toán Custom Smoothed Class Weights cho XGBoost...")
    
    unique_classes = np.unique(y_train)
    raw_class_weights = compute_class_weight('balanced', classes=unique_classes, y=y_train)
    weights_dict = {c: np.sqrt(w) for c, w in zip(unique_classes, raw_class_weights)}
    
    # tinh chỉnh weight thủ công
    if 4 in weights_dict: weights_dict[4] *= 0.6
    if 6 in weights_dict: weights_dict[6] *= 1.5
    if 9 in weights_dict: weights_dict[9] *= 2.5
    if 1 in weights_dict: weights_dict[1] *= 0.8

    # chuyển đổi thành mảng Sample Weight cho tập train
    final_sample_weights = np.array([weights_dict[y] for y in y_train])

    # train model xgboost
    xgb_model = xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.08,
        max_depth=5, 
        min_child_weight=2,
        gamma=1.0,
        reg_lambda=2.0,
        reg_alpha=0.5,
        subsample=0.75,         
        colsample_bytree=0.7,
        max_delta_step=3,
        random_state=42,
        eval_metric='mlogloss',
        tree_method="hist",
        device="cuda",  
        early_stopping_rounds=40
    )
    
    # dọn VRAM
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Đang huấn luyện XGBoost...")
    try:
        xgb_model.fit(
            X_train_emb, y_train,
            sample_weight=final_sample_weights,
            eval_set=[(X_train_emb, y_train), (X_valid_emb, y_valid)],
            verbose=100 
        )
    except Exception as e:
        print(f"Lỗi XGBoost, thử fallback về CPU... Lỗi: {e}")
        xgb_model.set_params(device='cpu')
        xgb_model.fit(
            X_train_emb, y_train,
            sample_weight=final_sample_weights,
            eval_set=[(X_train_emb, y_train), (X_valid_emb, y_valid)],
            verbose=100 
        )
        
    print(f"Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: {xgb_model.best_iteration}")
    
    # test trên tập test
    print("\n--- ĐÁNH GIÁ TRÊN TẬP TEST MASK ---")
    y_pred = xgb_model.predict(X_test_emb)
    
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    
    print(f"Accuracy:            {acc*100:.2f}%")
    print(f"F1-Score (Macro):    {f1_macro * 100:.2f}%")
    print(f"F1-Score (Weighted): {f1_weighted * 100:.2f}%\n")
    
    print("Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))
    
    return xgb_model

In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.utils.class_weight import compute_class_weight
import os
import gc
import torch

def get_stratified_tscv_indices(y, n_splits=5):
    """
    Kế thừa hàm chia OOF của CNN-BiLSTM để đảm bảo 2 mô hình sử dụng 
    TUYỆT ĐỐI CÙNG MỘT TẬP TRAIN/VALID TRONG TỪNG FOLD.
    """
    tscv = TimeSeriesSplit(n_splits=n_splits)
    train_folds = [[] for _ in range(n_splits)]
    val_folds = [[] for _ in range(n_splits)]
    
    classes = np.unique(y)
    for c in classes:
        c_indices = np.where(y == c)[0]
        for fold, (tr_idx_rel, val_idx_rel) in enumerate(tscv.split(c_indices)):
            train_folds[fold].extend(c_indices[tr_idx_rel])
            val_folds[fold].extend(c_indices[val_idx_rel])
            
    for fold in range(n_splits):
        train_folds[fold] = np.sort(train_folds[fold])
        val_folds[fold] = np.sort(val_folds[fold])
        
    return train_folds, val_folds

# --- KHỞI TẠO TÀI NGUYÊN PHA 1 CHO GAT + XGBOOST ---
N_SPLITS = 5
NUM_CLASSES = len(train_df['label'].unique())
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Tạo khung chứa mảng OOF toàn cục. Dùng np.nan để sau này dễ đồng bộ với NaNs của CNN_LSTM
meta_X_train_xgb = np.full((len(train_df), NUM_CLASSES), np.nan)

# Lấy chung 1 Split Index như bên DL
train_folds_idx, val_folds_idx = get_stratified_tscv_indices(train_df['label'].values, n_splits=N_SPLITS)

print("=== BẮT ĐẦU GIAI ĐOẠN 1: HUẤN LUYỆN OOF BẰNG GAT + XGBOOST ===")

for fold in range(N_SPLITS):
    print(f"\n{'='*20} FOLD {fold+1}/{N_SPLITS} {'='*20}")
    tr_idx = train_folds_idx[fold]
    val_idx = val_folds_idx[fold]
    
    # 1. TẠO DATA CỤC BỘ: Kéo tr_idx và val_idx, Sort thời gian để đồ thị căng dây đúng
    fold_idx = np.sort(np.concatenate([tr_idx, val_idx]))
    fold_df = train_df.iloc[fold_idx].copy().reset_index(drop=True)
    
    # 2. XÂY DỰNG SPLIT_TAG BẢO VỆ LEAKAGE: 
    # Lưu lại chỉ số gốc để tí Map OOF
    fold_df['orig_idx'] = fold_idx
    fold_df['split_tag'] = 0 # Mặc định là Train (nối 2 chiều)
    
    # Đánh dấu những Node thuộc tập Valid
    fold_df.loc[fold_df['orig_idx'].isin(val_idx), 'split_tag'] = 1
    
    # 3. Tạo Mask Động (Local Graph Masks)
    train_mask = torch.tensor((fold_df['split_tag'] == 0).values, dtype=torch.bool)
    valid_mask = torch.tensor((fold_df['split_tag'] == 1).values, dtype=torch.bool)
    test_mask = torch.zeros(len(fold_df), dtype=torch.bool) # OOF Fold nội bộ tuyệt đối k có Test
    
    # Bóc lưu index Valid gốc, rồi xóa cột để tránh lọt vào Node Features
    fold_orig_valid_idx = fold_df.loc[fold_df['split_tag'] == 1, 'orig_idx'].values
    fold_df_clean = fold_df.drop(columns=['split_tag', 'orig_idx'])
    
    # 4. CĂNG LƯỚI SUB-GRAPH NỘI BỘ FOLD
    print("\n[+] Đang xây dựng Sub-Graph cục bộ cho Fold...")
    graph = build_ip_topology_graph(fold_df_clean, train_mask, valid_mask, test_mask, window_size=10)
    
    # 5. Dataloaders cho GAT
    train_loader = NeighborLoader(graph, num_neighbors=[10, 5], input_nodes=graph.train_mask, batch_size=2048, shuffle=True, num_workers=0)
    valid_loader = NeighborLoader(graph, num_neighbors=[10, 5], input_nodes=graph.valid_mask, batch_size=2048, shuffle=False, num_workers=0)
    
    # BỔ SUNG LỚN: TÍNH TOÁN CLASS WEIGHTS CHO GAT DO DATASET QUÁ IMBALANCED
    print("\n[+] Đang tính toán Class Weights dành riêng cho GAT Embedder...")
    y_train_fold = fold_df_clean.loc[train_mask.numpy(), 'label'].values
    unique_classes_fold = np.unique(y_train_fold)
    raw_weights = compute_class_weight('balanced', classes=unique_classes_fold, y=y_train_fold)
    
    # Smooth weight giống như đã làm bên XGBoost
    weights_dict = {c: np.sqrt(w) for c, w in zip(unique_classes_fold, raw_weights)}
    class_weights_tensor = torch.ones(NUM_CLASSES, dtype=torch.float32)
    for c, w in weights_dict.items():
        class_weights_tensor[int(c)] = w
    class_weights_tensor = torch.clamp(class_weights_tensor, min=0.1, max=10.0) # Clip mượt trọng số
    class_weights_tensor = class_weights_tensor.to(DEVICE)
    
    # 6. Khởi tạo và Train GAT Embedder (SỬ DỤNG PARAMETER GIỐNG HỆT NOTEBOOK CŨ ĐẠT F1 0.93)
    print("\n[+] Đang huấn luyện GAT Embedder...")
    num_features = graph.x.shape[1]
    model_gat = GAT_Embedder(in_channels=num_features, hidden_channels=128, embedding_dim=64, num_classes=NUM_CLASSES, heads=8, edge_dropout=0.3, edge_dim=4).to(DEVICE)
    
    # lr=0.0005 giống hệ setup cũ (Tránh Gradient nổ như lúc set 0.005)
    model_gat = train_gat(model_gat, train_loader, valid_loader, device=DEVICE, epochs=20, lr=0.0005, patience=6, class_weights=class_weights_tensor)
    
    # 7. Trích xuất Embeddings 
    print("\n[+] Trích xuất Embeddings...")
    X_train_emb, y_train_emb = extract_embeddings_mask(model_gat, graph, mask=graph.train_mask, device=DEVICE)
    X_valid_emb, y_valid_emb = extract_embeddings_mask(model_gat, graph, mask=graph.valid_mask, device=DEVICE)
    
    # 8. Huấn luyện XGBoost trên cặp Embeddings
    print("\n[+] Huấn luyện cấu phần XGBoost...")
    # Truyền valid_emb vào vị trí của test_emb để hàm train_evaluate_xgboost không báo lỗi (Dummy Test)
    xgb_model = train_evaluate_xgboost(X_train_emb, y_train_emb, X_valid_emb, y_valid_emb, X_valid_emb, y_valid_emb)
    
    # 9. SINH DỰ ĐOÁN OOF PROBABILITIES
    print("\n[+] Lưu dự đoán XGBoost OOF vào ma trận...")
    valid_preds_proba = xgb_model.predict_proba(X_valid_emb)
    
    # MẢNH GHÉP QUYẾT ĐỊNH: Map chính xác kết quả vào dòng tương ứng của DataFrame gốc
    meta_X_train_xgb[fold_orig_valid_idx, :] = valid_preds_proba
    
    # ==========================================
    # NHẤT ĐỊNH PHẢI DỌN RÁC CHỐNG NGẬP RAM/VRAM
    # ==========================================
    del graph, train_loader, valid_loader, fold_df, fold_df_clean, model_gat, xgb_model
    del X_train_emb, y_train_emb, X_valid_emb, y_valid_emb, valid_preds_proba, fold_orig_valid_idx
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
print("\n=== HOÀN TẤT GIAI ĐOẠN 1: XUẤT FILE PARQUET OOF CHO XGBOOST ===")

# Đóng gói và lưu giữ nguyên Pandas Index gốc để gộp với CNN-LSTM
xgb_oof_columns = [f"xgb_class_{i}" for i in range(NUM_CLASSES)]
meta_X_train_xgb_df = pd.DataFrame(meta_X_train_xgb, columns=xgb_oof_columns, index=train_df.index)
meta_X_train_xgb_df['label'] = train_df['label'].values

os.makedirs("ip-files-train-meta", exist_ok=True)
xgb_oof_save_path = "ip-files-train-meta/meta_X_train_xgb.parquet"
meta_X_train_xgb_df.to_parquet(xgb_oof_save_path, engine="pyarrow")
print(f"Đã lưu Meta Features OOF của GAT-XGBoost tại: {xgb_oof_save_path}")

=== BẮT ĐẦU GIAI ĐOẠN 1: HUẤN LUYỆN OOF BẰNG GAT + XGBOOST ===

==================== FOLD 1/5 ====================

[+] Đang xây dựng Sub-Graph cục bộ cho Fold...
Bước 1: Trích xuất Node Features (X) và Label (y)...
Bước 1.5: Áp dụng QuantileTransformer (Chỉ fit trên Train Mask để chống Data Leakage)...
Bước 2: Xây Dựng Đồ Thị Dây Chuyền Theo IP (Temporal IP Topology)...


[1/2] Gom nhóm Gói tin theo IP:   0%|          | 0/823558 [00:00<?, ?it/s]

[2/2] Căng lưới đồ thị (Window = 10):   0%|          | 0/19 [00:00<?, ?it/s]


Hợp nhất và tạo Cạnh (Culling Edge Index)...
HOÀN THIỆN ĐỒ THỊ TEMPORAL IP (CÓ EDGE FEATURES)!
   - Số Nodes : 823,558
   - Số Edges : 21,457,704
   - Số Features/Cạnh : 4

[+] Đang tính toán Class Weights dành riêng cho GAT Embedder...

[+] Đang huấn luyện GAT Embedder...


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


...

=== Epoch 8 Tổng kết ===
   Train Loss: 1.2482 | Train Macro F1: 0.9838
   Valid Loss: 1.7896   | Valid Macro F1: 0.9507
   Learning Rate hiện tại: 0.000125
Báo động Early Stopping: Không cải thiện F1 6/6 epochs.

Đã kích hoạt Early Stopping tại Epoch 8! Lùi về checkpoint tốt nhất.

Hoàn tất Train! Đã load lại weights MANG LẠI F1 CAO NHẤT (Val Macro F1: 0.9631) để dùng trích xuất Embedding!

[+] Trích xuất Embeddings...
Đang khởi tạo Inference Loader lấy Embeddings...


Đang trích xuất Vectơ Embedding:   0%|          | 0/4022 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Embeddings: (2058868, 64)
Đang khởi tạo Inference Loader lấy Embeddings...


Đang trích xuất Vectơ Embedding:   0%|          | 0/805 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Embeddings: (411770, 64)

[+] Huấn luyện cấu phần XGBoost...

--- BẮT ĐẦU TRAIN XGBOOST TỪ BASE GAT EMBEDDING ---
Đang tính toán Custom Smoothed Class Weights cho XGBoost...
Đang huấn luyện XGBoost...
[0]	validation_0-mlogloss:1.56138	validation_1-mlogloss:1.57119
[100]	validation_0-mlogloss:0.00046	validation_1-mlogloss:0.10957
[200]	validation_0-mlogloss:0.00003	validation_1-mlogloss:0.06379
[215]	validation_0-mlogloss:0.00003	validation_1-mlogloss:0.06375
Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: 175

--- ĐÁNH GIÁ TRÊN TẬP TEST MASK ---
Accuracy:            99.00%
F1-Score (Macro):    96.74%
F1-Score (Weighted): 99.01%

Classification Report:
              precision    recall  f1-score   support

           0     0.9555    0.9720    0.9637     10748
           1     0.9992    1.0000    0.9996    262203
           2     0.9956    0.9993    0.9974      1360
           3     0.9993    1.0000    0.9997     19635
           4     

In [4]:
# chạy lại phase 1 với 3 fold đầu tiên để tạo ra file OOF cho 3 fold này
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.utils.class_weight import compute_class_weight
import os
import gc
import torch
os.makedirs("gat_xg_boost_oof", exist_ok=True)
def get_stratified_tscv_indices(y, n_splits=3):
    """
    Kế thừa hàm chia OOF của CNN-BiLSTM để đảm bảo 2 mô hình sử dụng 
    TUYỆT ĐỐI CÙNG MỘT TẬP TRAIN/VALID TRONG TỪNG FOLD.
    """
    tscv = TimeSeriesSplit(n_splits=n_splits)
    train_folds = [[] for _ in range(n_splits)]
    val_folds = [[] for _ in range(n_splits)]
    
    classes = np.unique(y)
    for c in classes:
        c_indices = np.where(y == c)[0]
        for fold, (tr_idx_rel, val_idx_rel) in enumerate(tscv.split(c_indices)):
            train_folds[fold].extend(c_indices[tr_idx_rel])
            val_folds[fold].extend(c_indices[val_idx_rel])
            
    for fold in range(n_splits):
        train_folds[fold] = np.sort(train_folds[fold])
        val_folds[fold] = np.sort(val_folds[fold])
        
    return train_folds, val_folds

# --- KHỞI TẠO TÀI NGUYÊN PHA 1 CHO GAT + XGBOOST ---
N_SPLITS = 3
NUM_CLASSES = len(train_df['label'].unique())
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Tạo khung chứa mảng OOF toàn cục. Dùng np.nan để sau này dễ đồng bộ với NaNs của CNN_LSTM
meta_X_train_xgb = np.full((len(train_df), NUM_CLASSES), np.nan)

# Lấy chung 1 Split Index như bên DL
train_folds_idx, val_folds_idx = get_stratified_tscv_indices(train_df['label'].values, n_splits=N_SPLITS)

print("=== BẮT ĐẦU GIAI ĐOẠN 1: HUẤN LUYỆN OOF BẰNG GAT + XGBOOST ===")

for fold in range(N_SPLITS):
    print(f"\n{'='*20} FOLD {fold+1}/{N_SPLITS} {'='*20}")
    tr_idx = train_folds_idx[fold]
    val_idx = val_folds_idx[fold]
    
    # 1. TẠO DATA CỤC BỘ: Kéo tr_idx và val_idx, Sort thời gian để đồ thị căng dây đúng
    fold_idx = np.sort(np.concatenate([tr_idx, val_idx]))
    fold_df = train_df.iloc[fold_idx].copy().reset_index(drop=True)
    
    # 2. XÂY DỰNG SPLIT_TAG BẢO VỆ LEAKAGE: 
    # Lưu lại chỉ số gốc để tí Map OOF
    fold_df['orig_idx'] = fold_idx
    fold_df['split_tag'] = 0 # Mặc định là Train (nối 2 chiều)
    
    # Đánh dấu những Node thuộc tập Valid
    fold_df.loc[fold_df['orig_idx'].isin(val_idx), 'split_tag'] = 1
    
    # 3. Tạo Mask Động (Local Graph Masks)
    train_mask = torch.tensor((fold_df['split_tag'] == 0).values, dtype=torch.bool)
    valid_mask = torch.tensor((fold_df['split_tag'] == 1).values, dtype=torch.bool)
    test_mask = torch.zeros(len(fold_df), dtype=torch.bool) # OOF Fold nội bộ tuyệt đối k có Test
    
    # Bóc lưu index Valid gốc, rồi xóa cột để tránh lọt vào Node Features
    fold_orig_valid_idx = fold_df.loc[fold_df['split_tag'] == 1, 'orig_idx'].values
    fold_df_clean = fold_df.drop(columns=['split_tag', 'orig_idx'])
    
    # 4. CĂNG LƯỚI SUB-GRAPH NỘI BỘ FOLD
    print("\n[+] Đang xây dựng Sub-Graph cục bộ cho Fold...")
    graph = build_ip_topology_graph(fold_df_clean, train_mask, valid_mask, test_mask, window_size=10)
    
    # 5. Dataloaders cho GAT
    train_loader = NeighborLoader(graph, num_neighbors=[10, 5], input_nodes=graph.train_mask, batch_size=2048, shuffle=True, num_workers=0)
    valid_loader = NeighborLoader(graph, num_neighbors=[10, 5], input_nodes=graph.valid_mask, batch_size=2048, shuffle=False, num_workers=0)
    
    # BỔ SUNG LỚN: TÍNH TOÁN CLASS WEIGHTS CHO GAT DO DATASET QUÁ IMBALANCED
    print("\n[+] Đang tính toán Class Weights dành riêng cho GAT Embedder...")
    y_train_fold = fold_df_clean.loc[train_mask.numpy(), 'label'].values
    unique_classes_fold = np.unique(y_train_fold)
    raw_weights = compute_class_weight('balanced', classes=unique_classes_fold, y=y_train_fold)
    
    # Smooth weight giống như đã làm bên XGBoost
    weights_dict = {c: np.sqrt(w) for c, w in zip(unique_classes_fold, raw_weights)}
    class_weights_tensor = torch.ones(NUM_CLASSES, dtype=torch.float32)
    for c, w in weights_dict.items():
        class_weights_tensor[int(c)] = w
    class_weights_tensor = torch.clamp(class_weights_tensor, min=0.1, max=10.0) # Clip mượt trọng số
    class_weights_tensor = class_weights_tensor.to(DEVICE)
    
    # 6. Khởi tạo và Train GAT Embedder (SỬ DỤNG PARAMETER GIỐNG HỆT NOTEBOOK CŨ ĐẠT F1 0.93)
    print("\n[+] Đang huấn luyện GAT Embedder...")
    num_features = graph.x.shape[1]
    model_gat = GAT_Embedder(in_channels=num_features, hidden_channels=128, embedding_dim=64, num_classes=NUM_CLASSES, heads=8, edge_dropout=0.3, edge_dim=4).to(DEVICE)
    
    # lr=0.0005 giống hệ setup cũ (Tránh Gradient nổ như lúc set 0.005)
    model_gat = train_gat(model_gat, train_loader, valid_loader, device=DEVICE, epochs=20, lr=0.0005, patience=4, class_weights=class_weights_tensor)
    # lưu model_gat sau khi train xong để sau này có thể load lại mà không cần train lại (Tiết kiệm thời gian cho 3 fold còn lại)
    torch.save(model_gat.state_dict(), f"gat_xg_boost_oof/gat_embedder_fold_{fold+1}.pth")

    # 7. Trích xuất Embeddings 
    print("\n[+] Trích xuất Embeddings...")
    X_train_emb, y_train_emb = extract_embeddings_mask(model_gat, graph, mask=graph.train_mask, device=DEVICE)
    X_valid_emb, y_valid_emb = extract_embeddings_mask(model_gat, graph, mask=graph.valid_mask, device=DEVICE)
    
    # 8. Huấn luyện XGBoost trên cặp Embeddings
    print("\n[+] Huấn luyện cấu phần XGBoost...")
    # Truyền valid_emb vào vị trí của test_emb để hàm train_evaluate_xgboost không báo lỗi (Dummy Test)
    xgb_model = train_evaluate_xgboost(X_train_emb, y_train_emb, X_valid_emb, y_valid_emb, X_valid_emb, y_valid_emb)
    # lưu model xgboost sau khi train xong để sau này có thể load lại mà không cần train lại (Tiết kiệm thời gian cho 3 fold còn lại)
    xgb_model.save_model(f"gat_xg_boost_oof/xgb_model_fold_{fold+1}.json")
    # 9. SINH DỰ ĐOÁN OOF PROBABILITIES
    print("\n[+] Lưu dự đoán XGBoost OOF vào ma trận...")
    valid_preds_proba = xgb_model.predict_proba(X_valid_emb)
    
    # MẢNH GHÉP QUYẾT ĐỊNH: Map chính xác kết quả vào dòng tương ứng của DataFrame gốc
    meta_X_train_xgb[fold_orig_valid_idx, :] = valid_preds_proba
    
    # ==========================================
    # NHẤT ĐỊNH PHẢI DỌN RÁC CHỐNG NGẬP RAM/VRAM
    # ==========================================
    del graph, train_loader, valid_loader, fold_df, fold_df_clean, model_gat, xgb_model
    del X_train_emb, y_train_emb, X_valid_emb, y_valid_emb, valid_preds_proba, fold_orig_valid_idx
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
print("\n=== HOÀN TẤT GIAI ĐOẠN 1: XUẤT FILE PARQUET OOF CHO XGBOOST ===")

# Đóng gói và lưu giữ nguyên Pandas Index gốc để gộp với CNN-LSTM
xgb_oof_columns = [f"xgb_class_{i}" for i in range(NUM_CLASSES)]
meta_X_train_xgb_df = pd.DataFrame(meta_X_train_xgb, columns=xgb_oof_columns, index=train_df.index)
meta_X_train_xgb_df['label'] = train_df['label'].values

os.makedirs("ip-files-train-meta", exist_ok=True)
xgb_oof_save_path = "ip-files-train-meta/meta_X_train_xgb.parquet"
meta_X_train_xgb_df.to_parquet(xgb_oof_save_path, engine="pyarrow")
print(f"Đã lưu Meta Features OOF của GAT-XGBoost tại: {xgb_oof_save_path}")

=== BẮT ĐẦU GIAI ĐOẠN 1: HUẤN LUYỆN OOF BẰNG GAT + XGBOOST ===

==================== FOLD 1/3 ====================

[+] Đang xây dựng Sub-Graph cục bộ cho Fold...
Bước 1: Trích xuất Node Features (X) và Label (y)...
Bước 1.5: Áp dụng QuantileTransformer (Chỉ fit trên Train Mask để chống Data Leakage)...
Bước 2: Xây Dựng Đồ Thị Dây Chuyền Theo IP (Temporal IP Topology)...


[1/2] Gom nhóm Gói tin theo IP:   0%|          | 0/1235322 [00:00<?, ?it/s]

[2/2] Căng lưới đồ thị (Window = 10):   0%|          | 0/21 [00:00<?, ?it/s]


Hợp nhất và tạo Cạnh (Culling Edge Index)...
HOÀN THIỆN ĐỒ THỊ TEMPORAL IP (CÓ EDGE FEATURES)!
   - Số Nodes : 1,235,322
   - Số Edges : 32,268,442
   - Số Features/Cạnh : 4

[+] Đang tính toán Class Weights dành riêng cho GAT Embedder...

[+] Đang huấn luyện GAT Embedder...


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


...

=== Epoch 6 Tổng kết ===
   Train Loss: 1.2482 | Train Macro F1: 0.9833
   Valid Loss: 1.7734   | Valid Macro F1: 0.9663
   Learning Rate hiện tại: 0.000250
Báo động Early Stopping: Không cải thiện F1 4/4 epochs.

Đã kích hoạt Early Stopping tại Epoch 6! Lùi về checkpoint tốt nhất.

Hoàn tất Train! Đã load lại weights MANG LẠI F1 CAO NHẤT (Val Macro F1: 0.9724) để dùng trích xuất Embedding!

[+] Trích xuất Embeddings...
Đang khởi tạo Inference Loader lấy Embeddings...


Đang trích xuất Vectơ Embedding:   0%|          | 0/3620 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Embeddings: (1852980, 64)
Đang khởi tạo Inference Loader lấy Embeddings...


Đang trích xuất Vectơ Embedding:   0%|          | 0/1207 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Embeddings: (617658, 64)

[+] Huấn luyện cấu phần XGBoost...

--- BẮT ĐẦU TRAIN XGBOOST TỪ BASE GAT EMBEDDING ---
Đang tính toán Custom Smoothed Class Weights cho XGBoost...
Đang huấn luyện XGBoost...
[0]	validation_0-mlogloss:1.56099	validation_1-mlogloss:1.57684
[100]	validation_0-mlogloss:0.00047	validation_1-mlogloss:0.04060
[131]	validation_0-mlogloss:0.00008	validation_1-mlogloss:0.04407
Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: 91

--- ĐÁNH GIÁ TRÊN TẬP TEST MASK ---
Accuracy:            99.04%
F1-Score (Macro):    93.51%
F1-Score (Weighted): 98.99%

Classification Report:
              precision    recall  f1-score   support

           0     0.9519    0.9829    0.9672     16122
           1     0.9946    1.0000    0.9973    393305
           2     1.0000    0.3787    0.5494      2041
           3     0.9977    1.0000    0.9989     29453
           4     0.9606    0.9162    0.9378     15416
           5     1.0000    0.9

In [4]:
import gc
import torch
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import QuantileTransformer

# --- KHỞI TẠO CÁC BIẾN MÔI TRƯỜNG ĐỂ CHẠY ĐỘC LẬP KHÔNG CẦN PHASE 1 ---
if 'NUM_CLASSES' not in locals():
    NUM_CLASSES = len(train_df['label'].unique())
if 'DEVICE' not in locals():
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
'''
# --- DỌN RÁC TRIỆT ĐỂ TỪ PHASE 1 CHỐNG TRÀN RAM ---
try:
    del meta_X_train_xgb_df, meta_X_train_xgb, train_folds_idx, val_folds_idx
    del fold_df, graph, train_loader, valid_loader, model_gat, xgb_model
    del X_train_emb, y_train_emb, X_valid_emb, y_valid_emb
except: pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
'''
print("=== BẮT ĐẦU GIAI ĐOẠN 2: HUẤN LUYỆN MODEL FINAL TRÊN TRAIN & TRÍCH XUẤT VALID + TEST ===")

print("\n[+] Đang gộp DataFrame toàn cục...")
# Đánh tag
train_df['split_tag'] = 0
valid_df['split_tag'] = 1 # Phase 2 giữ nguyên tập Valid để XGBoost dùng Early Stopping
test_df['split_tag'] = 2

# LƯU LẠI THÔNG TIN INDEX/LABEL ĐỂ XUẤT PARQUET
valid_df_index = valid_df.index.copy()
valid_df_label = valid_df['label'].copy()

test_df_index = test_df.index.copy()
test_df_label = test_df['label'].copy()

# =========================================================================
# LÕI QUAN TRỌNG NHẤT: Áp dụng QuantileTransformer CHO TOÀN BỘ DATA (Đồng bộ)
# =========================================================================
print("\n[+] Áp dụng QuantileTransformer (Feature Scaling đồng bộ OOF)...")
cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port', 'label', 'split_tag']
feature_cols = [c for c in train_df.columns if c not in cols_to_drop]

# Đảm bảo sử dụng toàn bộ mẫu (subsample=10000000) chống lệch distribution do chunk/sort
scaler_full = QuantileTransformer(output_distribution="normal", random_state=42)

print("   -> Đang Fit & Transform tập Train...")
train_df[feature_cols] = scaler_full.fit_transform(train_df[feature_cols].astype(np.float32))

print("   -> Đang Transform tập Valid & Test...")
valid_df[feature_cols] = scaler_full.transform(valid_df[feature_cols].astype(np.float32))
test_df[feature_cols]  = scaler_full.transform(test_df[feature_cols].astype(np.float32))
# =========================================================================

# Ghép nối Pandas
super_df = pd.concat([train_df, valid_df, test_df], ignore_index=True)

print("\n[-] XÓA DATAFRAME GỐC (TRAIN, VALID, TEST) ĐỂ GIẢI PHÓNG RAM...")
del train_df, valid_df, test_df
gc.collect()

print("\n[+] Sắp xếp Timestamp toàn cục...")
super_df.sort_values(by='timestamp_num', inplace=True)
super_df.reset_index(drop=True, inplace=True)

from sklearn.utils.class_weight import compute_class_weight

# Xây dựng Masks Toàn cục
train_mask = torch.tensor((super_df['split_tag'] == 0).values, dtype=torch.bool)
valid_mask = torch.tensor((super_df['split_tag'] == 1).values, dtype=torch.bool)
test_mask = torch.tensor((super_df['split_tag'] == 2).values, dtype=torch.bool)

# IN-PLACE DROP: Xóa biến trong DataFrame hiện tại chứ không tạo thêm bản copy
super_df.drop(columns=['split_tag'], inplace=True)
gc.collect()

print("\n[+] Đang xây dựng Đồ thị Cục Bộ toàn thể...")
super_graph = build_ip_topology_graph(super_df, train_mask, valid_mask, test_mask, window_size=10)

# MÃ HÓA GRAPH XONG, BIẾN DATA THÀNH TENSOR RỒI -> XÓA LUÔN BẢN PANDAS CUỐI CÙNG
print("\n[-] Xóa Super DataFrame giải phóng tiếp RAM...")
del super_df
gc.collect()

from torch_geometric.loader import NeighborLoader

# Dataloaders cho Mô hình Phân Loại Chính
train_loader_final = NeighborLoader(super_graph, num_neighbors=[10, 5], input_nodes=super_graph.train_mask, batch_size=2048, shuffle=True, num_workers=0)
valid_loader_final = NeighborLoader(super_graph, num_neighbors=[10, 5], input_nodes=super_graph.valid_mask, batch_size=2048, shuffle=False, num_workers=0)

# TÍNH TOÁN CLASS WEIGHTS CHO GAT
print("\n[+] Đang tính toán Class Weights dành riêng cho Final GAT Embedder...")
y_train_final = super_graph.y[super_graph.train_mask].cpu().numpy()
unique_classes_final = np.unique(y_train_final)
raw_weights_final = compute_class_weight('balanced', classes=unique_classes_final, y=y_train_final)

# Smooth weight 
weights_dict_final = {c: np.sqrt(w) for c, w in zip(unique_classes_final, raw_weights_final)}
class_weights_tensor_final = torch.ones(NUM_CLASSES, dtype=torch.float32)
for c, w in weights_dict_final.items():
    class_weights_tensor_final[int(c)] = w
class_weights_tensor_final = torch.clamp(class_weights_tensor_final, min=0.1, max=10.0)
class_weights_tensor_final = class_weights_tensor_final.to(DEVICE)

del y_train_final
gc.collect()

# GAT TRÍCH XUẤT FINAL EMBEDDING TOÀN CỤC CHỈ MỘT LẦN DUY NHẤT
print("\n[+] Khởi tạo và Huấn luyện Final GAT Model Toàn Cục...")
num_features = super_graph.x.shape[1]
final_gat = GAT_Embedder(in_channels=num_features, hidden_channels=128, embedding_dim=64, num_classes=NUM_CLASSES, heads=8, edge_dropout=0.3, edge_dim=4).to(DEVICE)
final_gat = train_gat(final_gat, train_loader_final, valid_loader_final, device=DEVICE, epochs=20, lr=0.0005, patience=4, class_weights=class_weights_tensor_final)

print("\n[+] Xuất Meta Môi Trường cho XGBoost Toàn Cục...")
X_train_super_emb, y_train_super_emb = extract_embeddings_mask(final_gat, super_graph, mask=super_graph.train_mask, device=DEVICE)
X_valid_super_emb, y_valid_super_emb = extract_embeddings_mask(final_gat, super_graph, mask=super_graph.valid_mask, device=DEVICE)
X_test_super_emb, y_test_super_emb = extract_embeddings_mask(final_gat, super_graph, mask=super_graph.test_mask, device=DEVICE)

print("\n[+] Huấn luyện XGBoost Toàn Cục (Dùng Valid làm Early Stopping)...")
final_xgb_model = train_evaluate_xgboost(X_train_super_emb, y_train_super_emb, X_valid_super_emb, y_valid_super_emb, X_test_super_emb, y_test_super_emb)

print("\n[+] Lấy Meta Features Holdout...")
meta_X_valid_xgb = final_xgb_model.predict_proba(X_valid_super_emb)
meta_X_test_xgb = final_xgb_model.predict_proba(X_test_super_emb)

print("\n=== HOÀN TẤT GIAI ĐOẠN 2: LƯU TRỮ METAFEATURES XGBOOST VALID & TEST ===")
os.makedirs("ip-files-train-meta", exist_ok=True)
os.makedirs("ip-files-test", exist_ok=True)

xgb_oof_columns = [f"xgb_class_{i}" for i in range(NUM_CLASSES)]

# Lưu Valid
xgb_valid_oof_save_path = "ip-files-train-meta/meta_X_valid_xgb.parquet"
meta_X_valid_xgb_df = pd.DataFrame(meta_X_valid_xgb, columns=xgb_oof_columns, index=valid_df_index)
meta_X_valid_xgb_df['label'] = valid_df_label.values
meta_X_valid_xgb_df.to_parquet(xgb_valid_oof_save_path, engine="pyarrow")
print(f"Đã lưu Meta Features GAT-XGBoost của tập VALID tại: {xgb_valid_oof_save_path}")

# Lưu Test
xgb_test_oof_save_path = "ip-files-test/meta_X_test_xgb.parquet"
meta_X_test_xgb_df = pd.DataFrame(meta_X_test_xgb, columns=xgb_oof_columns, index=test_df_index)
meta_X_test_xgb_df['label'] = test_df_label.values
meta_X_test_xgb_df.to_parquet(xgb_test_oof_save_path, engine="pyarrow")
print(f"Đã lưu Meta Features GAT-XGBoost của tập TEST tại:  {xgb_test_oof_save_path}")

del super_graph, train_loader_final, valid_loader_final
del final_gat, final_xgb_model, X_train_super_emb, y_train_super_emb
del X_valid_super_emb, y_valid_super_emb, X_test_super_emb, y_test_super_emb
if 'meta_X_test_xgb_df' in locals(): del meta_X_test_xgb_df
if 'meta_X_valid_xgb_df' in locals(): del meta_X_valid_xgb_df
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

=== BẮT ĐẦU GIAI ĐOẠN 2: HUẤN LUYỆN MODEL FINAL TRÊN TRAIN & TRÍCH XUẤT VALID + TEST ===

[+] Đang gộp DataFrame toàn cục...

[+] Áp dụng QuantileTransformer (Feature Scaling đồng bộ OOF)...
   -> Đang Fit & Transform tập Train...
   -> Đang Transform tập Valid & Test...

[-] XÓA DATAFRAME GỐC (TRAIN, VALID, TEST) ĐỂ GIẢI PHÓNG RAM...

[+] Sắp xếp Timestamp toàn cục...

[+] Đang xây dựng Đồ thị Cục Bộ toàn thể...
Bước 1: Trích xuất Node Features (X) và Label (y)...
Bước 2: Xây Dựng Đồ Thị Dây Chuyền Theo IP (Temporal IP Topology)...


[1/2] Gom nhóm Gói tin theo IP:   0%|          | 0/3801012 [00:00<?, ?it/s]

[2/2] Căng lưới đồ thị (Window = 10):   0%|          | 0/23 [00:00<?, ?it/s]


Hợp nhất và tạo Cạnh (Culling Edge Index)...
HOÀN THIỆN ĐỒ THỊ TEMPORAL IP (CÓ EDGE FEATURES)!
   - Số Nodes : 3,801,012
   - Số Edges : 107,556,942
   - Số Features/Cạnh : 4

[-] Xóa Super DataFrame giải phóng tiếp RAM...

[+] Đang tính toán Class Weights dành riêng cho Final GAT Embedder...

[+] Khởi tạo và Huấn luyện Final GAT Model Toàn Cục...


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


...

Trích xuất thành công! Kích thước Ma trận Embeddings: (570134, 64)
Đang khởi tạo Inference Loader lấy Embeddings...


Đang trích xuất Vectơ Embedding:   0%|          | 0/1485 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Embeddings: (760240, 64)

[+] Huấn luyện XGBoost Toàn Cục (Dùng Valid làm Early Stopping)...

--- BẮT ĐẦU TRAIN XGBOOST TỪ BASE GAT EMBEDDING ---
Đang tính toán Custom Smoothed Class Weights cho XGBoost...
Đang huấn luyện XGBoost...
[0]	validation_0-mlogloss:1.56093	validation_1-mlogloss:1.56453
[100]	validation_0-mlogloss:0.00050	validation_1-mlogloss:0.07265
[107]	validation_0-mlogloss:0.00032	validation_1-mlogloss:0.07664
Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: 67

--- ĐÁNH GIÁ TRÊN TẬP TEST MASK ---


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:751: UserWarning: [13:27:11] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Accuracy:            96.15%
F1-Score (Macro):    89.32%
F1-Score (Weighted): 96.09%

Classification Report:
              precision    recall  f1-score   support

           0     0.8131    0.9779    0.8879     19846
           1     0.9849    1.0000    0.9924    484077
           2     0.9423    1.0000    0.9703      2515
           3     1.0000    0.8123    0.8964     36253
           4     0.7332    0.8158    0.7723     18979
           5     1.0000    0.9992    0.9996      2451
           6     0.5931    0.7328    0.6556     11847
           7     1.0000    0.9934    0.9967    107436
           8     0.8075    0.9942    0.8912     16746
           9     0.9437    0.6670    0.7816     41523
          10     0.9997    0.9621    0.9806     18567

    accuracy                         0.9615    760240
   macro avg     0.8925    0.9050    0.8932    760240
weighted avg     0.9650    0.9615    0.9609    760240


[+] Lấy Meta Features Holdout...

=== HOÀN TẤT GIAI ĐOẠN 2: LƯU TRỮ METAFEATUR

In [14]:
print("TEST MODEL CŨ TRÊN DỮ LIỆU HIỆN TẠI...")

print("\n[+] Đang gộp DataFrame toàn cục...")
# Đánh tag
train_df['split_tag'] = 0
valid_df['split_tag'] = 1 # Phase 2 giữ nguyên tập Valid để XGBoost dùng Early Stopping
test_df['split_tag'] = 2

# LƯU LẠI THÔNG TIN INDEX/LABEL ĐỂ XUẤT PARQUET
valid_df_index = valid_df.index.copy()
valid_df_label = valid_df['label'].copy()

test_df_index = test_df.index.copy()
test_df_label = test_df['label'].copy()

# =========================================================================
# LÕI QUAN TRỌNG NHẤT: Áp dụng QuantileTransformer CHO TOÀN BỘ DATA (Đồng bộ)
# =========================================================================
print("\n[+] Áp dụng QuantileTransformer (Feature Scaling đồng bộ OOF)...")
cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port', 'label', 'split_tag']
feature_cols = [c for c in train_df.columns if c not in cols_to_drop]

# Đảm bảo sử dụng toàn bộ mẫu (subsample=10000000) chống lệch distribution do chunk/sort
scaler_full = QuantileTransformer(output_distribution="normal", random_state=42)

print("   -> Đang Fit & Transform tập Train...")
train_df[feature_cols] = scaler_full.fit_transform(train_df[feature_cols].astype(np.float32))

print("   -> Đang Transform tập Valid & Test...")
valid_df[feature_cols] = scaler_full.transform(valid_df[feature_cols].astype(np.float32))
test_df[feature_cols]  = scaler_full.transform(test_df[feature_cols].astype(np.float32))
# =========================================================================

# Ghép nối Pandas
super_df = pd.concat([train_df, valid_df, test_df], ignore_index=True)

print("\n[-] XÓA DATAFRAME GỐC (TRAIN, VALID, TEST) ĐỂ GIẢI PHÓNG RAM...")
del train_df, valid_df, test_df
gc.collect()

print("\n[+] Sắp xếp Timestamp toàn cục...")
super_df.sort_values(by='timestamp_num', inplace=True)
super_df.reset_index(drop=True, inplace=True)

from sklearn.utils.class_weight import compute_class_weight

# Xây dựng Masks Toàn cục
train_mask = torch.tensor((super_df['split_tag'] == 0).values, dtype=torch.bool)
valid_mask = torch.tensor((super_df['split_tag'] == 1).values, dtype=torch.bool)
test_mask = torch.tensor((super_df['split_tag'] == 2).values, dtype=torch.bool)

# IN-PLACE DROP: Xóa biến trong DataFrame hiện tại chứ không tạo thêm bản copy
super_df.drop(columns=['split_tag'], inplace=True)
gc.collect()

print("\n[+] Đang xây dựng Đồ thị Cục Bộ toàn thể...")
super_graph = build_ip_topology_graph(super_df, train_mask, valid_mask, test_mask, window_size=10)

# MÃ HÓA GRAPH XONG, BIẾN DATA THÀNH TENSOR RỒI -> XÓA LUÔN BẢN PANDAS CUỐI CÙNG
print("\n[-] Xóa Super DataFrame giải phóng tiếp RAM...")
del super_df
gc.collect()

TEST MODEL CŨ TRÊN DỮ LIỆU HIỆN TẠI...

[+] Đang gộp DataFrame toàn cục...

[+] Áp dụng QuantileTransformer (Feature Scaling đồng bộ OOF)...
   -> Đang Fit & Transform tập Train...
   -> Đang Transform tập Valid & Test...

[-] XÓA DATAFRAME GỐC (TRAIN, VALID, TEST) ĐỂ GIẢI PHÓNG RAM...

[+] Sắp xếp Timestamp toàn cục...

[+] Đang xây dựng Đồ thị Cục Bộ toàn thể...
Bước 1: Trích xuất Node Features (X) và Label (y)...
Bước 2: Xây Dựng Đồ Thị Dây Chuyền Theo IP (Temporal IP Topology)...


[1/2] Gom nhóm Gói tin theo IP:   0%|          | 0/3801012 [00:00<?, ?it/s]

[2/2] Căng lưới đồ thị (Window = 10):   0%|          | 0/23 [00:00<?, ?it/s]


Hợp nhất và tạo Cạnh (Culling Edge Index)...
HOÀN THIỆN ĐỒ THỊ TEMPORAL IP (CÓ EDGE FEATURES)!
   - Số Nodes : 3,801,012
   - Số Edges : 107,556,942
   - Số Features/Cạnh : 4

[-] Xóa Super DataFrame giải phóng tiếp RAM...


0

In [15]:
# load lai model GAT và XGBoost cũ để test lại trên tập test
# gat cũ: model_final\gat_embedder.pth
# xgboost cũ: model_final\GAT_XGB_Hybrid_FAISS_Model.json

gat_model = GAT_Embedder(in_channels=num_features, hidden_channels=128, embedding_dim=64, num_classes=NUM_CLASSES, heads=8, edge_dropout=0.3, edge_dim=4).to(DEVICE)
gat_model.load_state_dict(torch.load(r"model_final\gat_embedder.pth", map_location=DEVICE))
import xgboost as xgb
xgb_model = xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.08,
        max_depth=5, 
        min_child_weight=2,
        gamma=1.0,
        reg_lambda=2.0,
        reg_alpha=0.5,
        subsample=0.75,         
        colsample_bytree=0.7,
        max_delta_step=3,
        random_state=42,
        eval_metric='mlogloss',
        tree_method="hist",
        device="cuda",  
        early_stopping_rounds=40
    )
xgb_model.load_model(r"model_final\GAT_XGB_Hybrid_FAISS_Model.json")
print("\n[+] Đánh giá lại trên tập TEST bằng model GAT + XGBoost đã lưu...")
# Trích xuất embedding từ model GAT đã lưu
test_embeddings, test_labels = extract_embeddings_mask(gat_model, super_graph, mask=super_graph.test_mask, device=DEVICE)
# Dự đoán bằng XGBoost đã lưu
test_preds = xgb_model.predict(test_embeddings)
from sklearn.metrics import classification_report, accuracy_score, f1_score
acc = accuracy_score(test_labels, test_preds)
f1_macro = f1_score(test_labels, test_preds, average='macro')
f1_weighted = f1_score(test_labels, test_preds, average='weighted')
print(f"Accuracy:            {acc*100:.2f}%")
print(f"F1-Score (Macro):    {f1_macro * 100:.2f}%")
print(f"F1-Score (Weighted): {f1_weighted * 100:.2f}%\n")

C:\Users\Admin\AppData\Local\Temp\ipykernel_21036\3466069320.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  gat_model.load_state_dict(torch.load(r"model_final\gat_embed


[+] Đánh giá lại trên tập TEST bằng model GAT + XGBoost đã lưu...
Đang khởi tạo Inference Loader lấy Embeddings...


Đang trích xuất Vectơ Embedding:   0%|          | 0/1485 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Embeddings: (760240, 64)
Accuracy:            97.42%
F1-Score (Macro):    92.79%
F1-Score (Weighted): 97.35%



In [18]:
# dùng model cũ để in tạo ra meta_xgb_test và meta_xgb_valid rồi lưu lại để so sánh với meta_xgb_test và meta_xgb_valid mới tạo ở trên
print("\n[+] Sinh dự đoán OOF cho tập VALID và TEST bằng model GAT + XGBoost đã lưu...")
# Dự đoán OOF cho tập VALID
valid_embeddings, valid_labels = extract_embeddings_mask(gat_model, super_graph, mask=super_graph.valid_mask, device=DEVICE)
valid_oof_preds = xgb_model.predict_proba(valid_embeddings)
# Dự đoán OOF cho tập TEST
test_oof_preds = xgb_model.predict_proba(test_embeddings)

# lưu meta_xgb_valid và meta_xgb_test vào file parquet để so sánh với meta_xgb_valid và meta_xgb_test mới tạo ở trên
xgb_oof_columns = [f"xgb_class_{i}" for i in range(NUM_CLASSES)]
# Lưu Valid
xgb_valid_oof_save_path = "ip-files-train-meta/meta_X_valid_xgb_old.parquet"
meta_X_valid_xgb_old_df = pd.DataFrame(valid_oof_preds, columns=xgb_oof_columns, index=valid_df_index)
meta_X_valid_xgb_old_df['label'] = valid_df_label.values
meta_X_valid_xgb_old_df.to_parquet(xgb_valid_oof_save_path, engine="pyarrow")
print(f"Đã lưu Meta Features GAT-XGBoost cũ của tập VALID tại: {xgb_valid_oof_save_path}")

# dự đoán oof cho tập TEST
xgb_test_oof_save_path = "ip-files-test/meta_X_test_xgb_old.parquet"
meta_X_test_xgb_old_df = pd.DataFrame(test_oof_preds, columns=xgb_oof_columns, index=test_df_index)
meta_X_test_xgb_old_df['label'] = test_df_label.values
meta_X_test_xgb_old_df.to_parquet(xgb_test_oof_save_path, engine="pyarrow")
print(f"Đã lưu Meta Features GAT-XGBoost cũ của tập TEST tại:  {xgb_test_oof_save_path}")


[+] Sinh dự đoán OOF cho tập VALID và TEST bằng model GAT + XGBoost đã lưu...
Đang khởi tạo Inference Loader lấy Embeddings...


Đang trích xuất Vectơ Embedding:   0%|          | 0/1114 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Embeddings: (570134, 64)
Đã lưu Meta Features GAT-XGBoost cũ của tập VALID tại: ip-files-train-meta/meta_X_valid_xgb_old.parquet
Đã lưu Meta Features GAT-XGBoost cũ của tập TEST tại:  ip-files-test/meta_X_test_xgb_old.parquet
